# Tokenization Scripts
## 1- BPE Tokenization

### Loading Libraries

In [12]:
from collections import defaultdict
import re
import json
from typing import List, Tuple, Dict
import os
from pathlib import Path
import PyPDF2

### Reading Transcrip file

In [18]:
def read_transcript_file(file_path):
    """
    Reading file with transcript from investor call.
    """
    with open(file_path, 'rb') as file:
        pdf_reader = PyPDF2.PdfReader(file)
        full_text = ""
        
        for page_num in range(len(pdf_reader.pages)):
            page = pdf_reader.pages[page_num]
            text = page.extract_text()
            full_text += text + "\n"
            
        return full_text


def clean_transcript_text(text):
    text = re.sub(r'={5,}\s*Page\s*\d+\s*={5,}', '', text)
    text = re.sub(r'\n\s*\n', '\n\n', text)
    text = re.sub(r'\x0c', '', text)
    text = re.sub(r' {2,}', ' ', text)
    return text.strip()

def extract_sections(text):
    """
    Extract different sections from the transcript
    """
    sections = {}
    
    presentation_match = re.search(r'PRESENTATION\s*(.*?)(?=QUESTIONS AND ANSWERS|Q&A|$)', text, re.DOTALL | re.IGNORECASE)
    if presentation_match:
        sections['presentation'] = presentation_match.group(1).strip()
    
    qa_match = re.search(r'(?:QUESTIONS AND ANSWERS|Q&A)\s*(.*?)(?=Operator.*?concluded|$)', text, re.DOTALL | re.IGNORECASE)
    if qa_match:
        sections['qa'] = qa_match.group(1).strip()
    
    participants_match = re.search(r'PARTICIPANTS\s*(.*?)(?=PRESENTATION|$)', text, re.DOTALL | re.IGNORECASE)
    if participants_match:
        sections['participants'] = participants_match.group(1).strip()
    
    return sections

def extract_speaker_content(text, speaker_name):
    """
    Extract content spoken by a specific speaker - this is specific to Refinitiv.
    """
    pattern = rf'{speaker_name}.*?\n(.*?)(?=\n\s*[A-Z][a-z]+ [A-Z][a-z]+|$)'
    matches = re.findall(pattern, text, re.DOTALL | re.IGNORECASE)
    return '\n'.join(matches).strip()

def get_transcript_statistics(text):
    """
    Get basic statistics about the transcript.
    """
    english_words = re.findall(r'[a-zA-Z]+', text)
    chinese_chars = re.findall(r'[\u4e00-\u9fff]', text)
    
    stats = {
        'total_characters': len(text),
        'total_words': len(english_words),
        'chinese_characters': len(chinese_chars),
        'total_lines': len(text.split('\n')),
        'unique_english_words': len(set(english_words)),
        'unique_chinese_chars': len(set(chinese_chars))
    }
    return stats

def language_ratio(text):
    """
    Calculate the ratio of English to Chinese in the text.
    """
    english = len(re.findall(r'[a-zA-Z]', text))
    chinese = len(re.findall(r'[\u4e00-\u9fff]', text))
    total = english + chinese
    return {
        'english_pct': (english/total)*100 if total > 0 else 0,
        'chinese_pct': (chinese/total)*100 if total > 0 else 0
    }

file_path = "./files/Luckin Coffee First Quarter 2026 Earnings Transcript.pdf"

print("READING LUCKIN COFFEE EARNINGS TRANSCRIPT")
print("=" * 70)

full_text = read_transcript_file(file_path)

if full_text:
    print(f"Successfully read file: {os.path.basename(file_path)}")
    print(f"Raw file size: {len(full_text):,} characters")
    print(f"Number of pages: {len(full_text.split('Page')) - 1}")
    
    cleaned_text = clean_transcript_text(full_text)
    print(f"After cleaning: {len(cleaned_text):,} characters")
    
    stats = get_transcript_statistics(cleaned_text)
    print("\nTRANSCRIPT STATISTICS:")
    print("-" * 40)
    for key, value in stats.items():
        print(f"  {key.replace('_', ' ').title()}: {value:,}")
    
    lang_ratio = language_ratio(cleaned_text)
    print(f"\n  English content: {lang_ratio['english_pct']:.1f}%")
    print(f"  Chinese content: {lang_ratio['chinese_pct']:.1f}%")
    
    sections = extract_sections(cleaned_text)
    
    print("\nSECTIONS EXTRACTED:")
    print("-" * 40)
    for section_name, section_content in sections.items():
        word_count = len(section_content.split())
        print(f"  - {section_name.title()}: {word_count:,} words")
    
    print("\nSPEAKER CONTENT EXAMPLES:")
    print("-" * 40)
    
    speakers = ["Jinyi Guo", "Jing An", "Nancy Song"]
    for speaker in speakers:
        content = extract_speaker_content(cleaned_text, speaker)
        if content:
            first_100 = content[:150].replace('\n', ' ')
            print(f"\n  {speaker}:")
            print(f"    {first_100}...")
    
    output_file = "./files/luckin_coffee_transcript_cleaned.txt"
    with open(output_file, 'w', encoding='utf-8') as f:
        f.write(cleaned_text)
    print(f"\nCleaned transcript saved to: {output_file}")
    

    
    print("\n" + "=" * 70)
    print("FULL TEXT PREVIEW (first 1000 characters):")
    print("=" * 70)
    print(cleaned_text[:1000])
    print("\n... (truncated)")
    print("=" * 70)
    
    print("\nREADY TO USE THIS TEXT FOR BPE TRAINING")
    print("=" * 70)
    
else:
    print("Failed to read the PDF file. Check that PyPDF2 is installed:")
    print("  pip install PyPDF2")
    print(f"  Current path: {file_path}")

READING LUCKIN COFFEE EARNINGS TRANSCRIPT
Successfully read file: Luckin Coffee First Quarter 2026 Earnings Transcript.pdf
Raw file size: 30,657 characters
Number of pages: 0
After cleaning: 30,577 characters

TRANSCRIPT STATISTICS:
----------------------------------------
  Total Characters: 30,577
  Total Words: 3,731
  Chinese Characters: 3,353
  Total Lines: 446
  Unique English Words: 961
  Unique Chinese Chars: 584

  English content: 85.6%
  Chinese content: 14.4%

SECTIONS EXTRACTED:
----------------------------------------
  - Presentation: 2,654 words
  - Qa: 0 words
  - Participants: 29 words

SPEAKER CONTENT EXAMPLES:
----------------------------------------

  Jinyi Guo:
    Jing An CFO, Luckin Coffee  Co-founder and CEO, Luckin Coffee  (Translated) Good day, everyone, and welcome to today’s earnings conference call. Thank...

  Jing An:
    Nancy Song Investor Relations Director , Luckin Coffee  CFO, Luckin Coffee...

  Nancy Song:
    Jessie Xu Analyst, J.P. Morgan  Rela

### Byte-pair Encoding Without Librairy

In [22]:
import re
from collections import defaultdict

def load_transcript():
    with open('./files/luckin_coffee_transcript_cleaned.txt', 'r', encoding='utf-8') as f:
        return f.read()

def build_initial_vocab(text):
    words = re.findall(r'[a-zA-Z]+|[0-9,]+[.]?[0-9]*', text)
    word_freq = defaultdict(int)
    for w in words:
        if len(w) > 0:
            word_freq[w] += 1
    
    vocab = {}
    for word, freq in word_freq.items():
        vocab[' '.join(list(word)) + ' </w>'] = freq
    return vocab

def count_pairs(vocab):
    pairs = defaultdict(int)
    for word, freq in vocab.items():
        symbols = word.split()
        for i in range(len(symbols) - 1):
            pairs[(symbols[i], symbols[i+1])] += freq
    return pairs

def perform_merge(pair, vocab):
    new_vocab = {}
    bigram = ' '.join(pair)
    merged = ''.join(pair)
    for word, freq in vocab.items():
        new_word = word.replace(bigram, merged)
        new_vocab[new_word] = freq
    return new_vocab

def tokenize_word(word, merges):
    tokens = list(word) + ['</w>']
    
    for merge in merges:
        i = 0
        new_tokens = []
        while i < len(tokens):
            if (i < len(tokens) - 1 and 
                tokens[i] == merge[0] and 
                tokens[i+1] == merge[1]):
                new_tokens.append(''.join(merge))
                i += 2
            else:
                new_tokens.append(tokens[i])
                i += 1
        tokens = new_tokens
    
    if tokens and tokens[-1] == '</w>':
        tokens = tokens[:-1]
    return tokens

text = load_transcript()
print(f"Loaded {len(text):,} characters\n")

vocab = build_initial_vocab(text)
print(f"Initial vocabulary: {len(vocab)} character-level tokens")

merges = []
max_merges = 100

print("\nTraining BPE...")
print("-" * 50)

for i in range(max_merges):
    pairs = count_pairs(vocab)
    if not pairs:
        break
    
    best_pair = max(pairs, key=pairs.get)
    freq = pairs[best_pair]
    
    vocab = perform_merge(best_pair, vocab)
    merges.append(best_pair)
    
    # Show progress at key intervals
    if i < 30 and freq > 15:
        merged = ''.join(best_pair)
        print(f"Merge {i+1:3d}: {best_pair[0]} + {best_pair[1]} -> {merged:12} (freq: {freq:4d})")
    elif i % 20 == 0 and i > 0:
        print(f"Merge {i+1:3d}: Vocabulary now {len(vocab)} tokens")

print(f"\nTotal merges: {len(merges)}")
print(f"Final vocabulary: {len(vocab)} tokens")

print("\n" + "=" * 60)
print("TOKENIZATION RESULTS")
print("=" * 60)

test_words = [
    'revenue', 'operating', 'profit', 'margin', 
    'growth', 'store', 'delivery', 'customer',
    'RMB12', '33596', '2531', 'billion', 'million'
]

print("\nWord -> Tokens")
print("-" * 50)
for word in test_words:
    tokens = tokenize_word(word, merges)
    print(f"{word:15} -> {' + '.join(tokens)}")

print("\n" + "=" * 60)
print("WHAT BPE LEARNED FROM LUCKIN COFFEE TRANSCRIPT")
print("=" * 60)

# Find meaningful patterns from the actual merges
word_endings = []
common_prefixes = []
number_combos = []
financial_terms = []

for merge in merges[:50]:
    combined = ''.join(merge)
    if combined.endswith('</w>') and len(combined) > 3:
        word_endings.append(combined.replace('</w>', ''))
    elif any(c.isdigit() for c in combined):
        number_combos.append(combined)
    elif combined in ['re', 'ing', 'ion', 'at', 'st', 'er', 'in', 'on']:
        common_prefixes.append(combined)
    elif combined in ['store', 'market', 'growth', 'profit']:
        financial_terms.append(combined)

if word_endings:
    print("\n1. Common word endings learned:")
    for ending in word_endings[:5]:
        print(f"   - {ending}")

if common_prefixes:
    print("\n2. Common letter combinations:")
    for combo in common_prefixes[:5]:
        print(f"   - {combo}")

if number_combos:
    print("\n3. Number patterns learned:")
    for combo in number_combos[:3]:
        print(f"   - {combo}")

# Show which merges actually created meaningful financial terms
print("\n" + "=" * 60)
print("HOW BPE BUILDS FINANCIAL TERMS")
print("=" * 60)

print("\nExample: Building 'revenue'")
print("-" * 40)
tokens = list('revenue') + ['</w>']
print(f"Start: {tokens}")
for i, merge in enumerate(merges[:15], 1):
    new_tokens = []
    j = 0
    while j < len(tokens):
        if (j < len(tokens) - 1 and 
            tokens[j] == merge[0] and 
            tokens[j+1] == merge[1]):
            new_tokens.append(''.join(merge))
            j += 2
        else:
            new_tokens.append(tokens[j])
            j += 1
    tokens = new_tokens
    if ''.join(merge) in ['er', 'en', 're', 'ev']:
        print(f"After merge {i}: {tokens}")

print(f"\nFinal: {' + '.join(tokens[:-1])}")

print("\n" + "=" * 60)
print("OBSERVATIONS FROM THIS TRANSCRIPT")
print("=" * 60)

# Analyze what the merges tell us about the text
freq_merges = []
for merge in merges[:30]:
    combined = ''.join(merge)
    if combined == 'e</w>':
        freq_merges.append(('e</w>', 'plural nouns, past tense'))
    elif combined == 's</w>':
        freq_merges.append(('s</w>', 'plural nouns, third person'))
    elif combined == 'ing</w>':
        freq_merges.append(('ing</w>', 'present participles'))
    elif combined == 'ed</w>':
        freq_merges.append(('ed</w>', 'past tense'))
    elif combined == 'ion':
        freq_merges.append(('ion', 'nouns from verbs'))

if freq_merges:
    print("\nThe BPE algorithm found these patterns in the Luckin Coffee transcript:")
    for pattern, use in freq_merges[:4]:
        print(f"   - '{pattern}' is used for {use}")

print("\nThis shows the transcript contains:")
print("   - Many plural nouns (e</w>, s</w> merges)")
print("   - Financial metrics (numbers, percentages)")
print("   - Business terminology (revenue, growth, operating)")
print("   - Present participles (ing</w> merge)")

Loaded 30,577 characters

Initial vocabulary: 1050 character-level tokens

Training BPE...
--------------------------------------------------
Merge   1: e + </w> -> e</w>        (freq:  669)
Merge   2: i + n -> in           (freq:  532)
Merge   3: s + </w> -> s</w>        (freq:  446)
Merge   4: e + r -> er           (freq:  396)
Merge   5: d + </w> -> d</w>        (freq:  390)
Merge   6: o + n -> on           (freq:  342)
Merge   7: a + n -> an           (freq:  312)
Merge   8: , + </w> -> ,</w>        (freq:  302)
Merge   9: t + h -> th           (freq:  247)
Merge  10: s + t -> st           (freq:  239)
Merge  11: e + n -> en           (freq:  231)
Merge  12: r + </w> -> r</w>        (freq:  231)
Merge  13: y + </w> -> y</w>        (freq:  207)
Merge  14: r + e -> re           (freq:  204)
Merge  15: g + </w> -> g</w>        (freq:  203)
Merge  16: o + u -> ou           (freq:  198)
Merge  17: a + l -> al           (freq:  197)
Merge  18: a + t -> at           (freq:  185)
Merge  19

In [33]:
from collections import defaultdict
import re

# ============================================================
# Investment Management Corpus
# ============================================================

corpus = """
investment investment investment investment
investor investor investor
investing investing
invested
reinvestment

investment management
investment committee
investment strategy
investment portfolio
investment process
investment performance
investment opportunity
investment thesis
"""

# ============================================================
# Step 1 - Build Initial Vocabulary
# ============================================================

def build_vocab(text):

    words = re.findall(r'\w+', text.lower())

    vocab = defaultdict(int)

    for word in words:
        chars = tuple(list(word) + ['</w>'])
        vocab[chars] += 1

    return vocab


# ============================================================
# Step 2 - Count Pair Frequencies
# ============================================================

def get_pair_stats(vocab):

    pairs = defaultdict(int)

    for word, freq in vocab.items():

        for i in range(len(word) - 1):
            pair = (word[i], word[i + 1])
            pairs[pair] += freq

    return pairs


# ============================================================
# Step 3 - Merge Most Frequent Pair
# ============================================================

def merge_pair(vocab, pair):

    merged_token = ''.join(pair)

    new_vocab = {}

    for word, freq in vocab.items():

        new_word = []

        i = 0

        while i < len(word):

            if (
                i < len(word) - 1
                and word[i] == pair[0]
                and word[i + 1] == pair[1]
            ):
                new_word.append(merged_token)
                i += 2

            else:
                new_word.append(word[i])
                i += 1

        new_vocab[tuple(new_word)] = freq

    return new_vocab


# ============================================================
# Build Vocabulary
# ============================================================

vocab = build_vocab(corpus)

print("INITIAL VOCABULARY")
print("=" * 60)

for word, freq in sorted(vocab.items(), key=lambda x: -x[1])[:20]:
    print(f"{''.join(word):20} freq={freq}")

# ============================================================
# Run BPE
# ============================================================

num_merges = 50

print("\n")
print("BPE TRAINING")
print("=" * 60)

merges = []

for step in range(num_merges):

    pair_stats = get_pair_stats(vocab)

    best_pair = max(pair_stats, key=pair_stats.get)

    merged = ''.join(best_pair)

    print(
        f"Merge {step+1:2d}: "
        f"{best_pair[0]} + {best_pair[1]}"
        f" -> {merged}"
        f" ({pair_stats[best_pair]} occurrences)"
    )

    merges.append(best_pair)

    vocab = merge_pair(vocab, best_pair)

# ============================================================
# Tokenize New Words Using Learned Merges
# ============================================================

def tokenize(word, merges):

    tokens = list(word) + ['</w>']

    for merge in merges:

        merged_token = ''.join(merge)

        i = 0
        new_tokens = []

        while i < len(tokens):

            if (
                i < len(tokens) - 1
                and tokens[i] == merge[0]
                and tokens[i + 1] == merge[1]
            ):
                new_tokens.append(merged_token)
                i += 2

            else:
                new_tokens.append(tokens[i])
                i += 1

        tokens = new_tokens

    return [t for t in tokens if t != '</w>']


# ============================================================
# Results
# ============================================================

print("\n")
print("TOKENIZATION RESULTS")
print("=" * 60)

test_words = [
    "investor",
    "investment",
    "investing",
    "invested",
    "reinvestment",
    "management"
]

for word in test_words:

    tokens = tokenize(word, merges)

    print(f"{word:30} -> {' + '.join(tokens)}")

INITIAL VOCABULARY
investment</w>       freq=12
investor</w>         freq=3
investing</w>        freq=2
invested</w>         freq=1
reinvestment</w>     freq=1
management</w>       freq=1
committee</w>        freq=1
strategy</w>         freq=1
portfolio</w>        freq=1
process</w>          freq=1
performance</w>      freq=1
opportunity</w>      freq=1
thesis</w>           freq=1


BPE TRAINING
Merge  1: i + n -> in (21 occurrences)
Merge  2: e + s -> es (21 occurrences)
Merge  3: in + v -> inv (19 occurrences)
Merge  4: inv + es -> inves (19 occurrences)
Merge  5: inves + t -> invest (19 occurrences)
Merge  6: m + e -> me (14 occurrences)
Merge  7: me + n -> men (14 occurrences)
Merge  8: men + t -> ment (14 occurrences)
Merge  9: ment + </w> -> ment</w> (14 occurrences)
Merge 10: invest + ment</w> -> investment</w> (13 occurrences)
Merge 11: o + r -> or (6 occurrences)
Merge 12: invest + or -> investor (3 occurrences)
Merge 13: investor + </w> -> investor</w> (3 occurrences)
Merge 1

### WordPiece tokenization

In [37]:
from collections import defaultdict
import re
import math

corpus = """
investment investment investment investment
investor investor investor
investing investing
invested
reinvestment

investment management
investment committee
investment strategy
investment portfolio
investment process
investment performance
investment opportunity
investment thesis
"""

def build_vocab(text):
    words = re.findall(r'\w+', text.lower())
    vocab = defaultdict(int)
    for word in words:
        chars = tuple(list(word) + ['</w>'])
        vocab[chars] += 1
    return vocab

def get_pair_stats(vocab):
    pairs = defaultdict(int)
    for word, freq in vocab.items():
        for i in range(len(word) - 1):
            pair = (word[i], word[i + 1])
            pairs[pair] += freq
    return pairs

def get_wordpiece_scores(vocab, pairs):
    """
    WordPiece uses a different scoring function.
    It scores each pair based on how much it increases the likelihood
    of the training data under the model.
    """
    # Count total tokens
    total_tokens = sum(len(word) for word in vocab.keys())
    
    scores = {}
    for pair, freq in pairs.items():
        # Get frequency of the merged token
        merged = ''.join(pair)
        merged_freq = 0
        for word, word_freq in vocab.items():
            word_str = ''.join(word)
            if merged in word_str:
                merged_freq += word_freq
        
        # WordPiece score: (frequency of pair) / (frequency of first token * frequency of second token)
        # This is the log-likelihood ratio
        first_freq = sum(1 for word in vocab.keys() if pair[0] in word)
        second_freq = sum(1 for word in vocab.keys() if pair[1] in word)
        
        if first_freq > 0 and second_freq > 0:
            # WordPiece maximizes the likelihood of the segmentation
            score = freq / (first_freq * second_freq)
            scores[pair] = score
    
    return scores

def merge_pair(vocab, pair):
    merged_token = ''.join(pair)
    new_vocab = {}
    for word, freq in vocab.items():
        new_word = []
        i = 0
        while i < len(word):
            if (
                i < len(word) - 1
                and word[i] == pair[0]
                and word[i + 1] == pair[1]
            ):
                new_word.append(merged_token)
                i += 2
            else:
                new_word.append(word[i])
                i += 1
        new_vocab[tuple(new_word)] = freq
    return new_vocab

vocab = build_vocab(corpus)

print("INITIAL VOCABULARY")
print("=" * 60)
for word, freq in sorted(vocab.items(), key=lambda x: -x[1])[:20]:
    print(f"{''.join(word):20} freq={freq}")

num_merges = 50
print("\n")
print("WORD PIECE TRAINING")
print("=" * 60)

merges = []

for step in range(num_merges):
    pairs = get_pair_stats(vocab)
    if not pairs:
        break
    
    # WordPiece: score pairs instead of just counting frequency
    scores = get_wordpiece_scores(vocab, pairs)
    best_pair = max(scores, key=scores.get)
    
    merged = ''.join(best_pair)
    print(
        f"Merge {step+1:2d}: "
        f"{best_pair[0]} + {best_pair[1]}"
        f" -> {merged}"
        f" (score: {scores[best_pair]:.4f})"
    )
    
    merges.append(best_pair)
    vocab = merge_pair(vocab, best_pair)

def tokenize(word, merges):
    tokens = list(word) + ['</w>']
    for merge in merges:
        merged_token = ''.join(merge)
        i = 0
        new_tokens = []
        while i < len(tokens):
            if (
                i < len(tokens) - 1
                and tokens[i] == merge[0]
                and tokens[i + 1] == merge[1]
            ):
                new_tokens.append(merged_token)
                i += 2
            else:
                new_tokens.append(tokens[i])
                i += 1
        tokens = new_tokens
    return [t for t in tokens if t != '</w>']

print("\n")
print("TOKENIZATION RESULTS")
print("=" * 60)

test_words = [
    "investor",
    "investment",
    "investing",
    "invested",
    "reinvestment",
    "management"
]

for word in test_words:
    tokens = tokenize(word, merges)
    print(f"{word:30} -> {' + '.join(tokens)}")

INITIAL VOCABULARY
investment</w>       freq=12
investor</w>         freq=3
investing</w>        freq=2
invested</w>         freq=1
reinvestment</w>     freq=1
management</w>       freq=1
committee</w>        freq=1
strategy</w>         freq=1
portfolio</w>        freq=1
process</w>          freq=1
performance</w>      freq=1
opportunity</w>      freq=1
thesis</w>           freq=1


WORD PIECE TRAINING
Merge  1: n + v -> nv (score: 0.4750)
Merge  2: i + nv -> inv (score: 0.4222)
Merge  3: inv + e -> inve (score: 0.3455)
Merge  4: inve + s -> inves (score: 0.4750)
Merge  5: inves + t -> invest (score: 0.3455)
Merge  6: invest + m -> investm (score: 0.5200)
Merge  7: investm + e -> investme (score: 0.7222)
Merge  8: investme + n -> investmen (score: 1.0833)
Merge  9: investmen + t -> investment (score: 0.8125)
Merge 10: investment + </w> -> investment</w> (score: 0.5000)
Merge 11: u + n -> un (score: 0.2500)
Merge 12: n + g -> ng (score: 0.2222)
Merge 13: i + ng -> ing (score: 0.4000)
Me

### Unigram Laguage Model

In [43]:
from collections import defaultdict
import re
import math

# Simpler corpus with clear patterns
corpus = """
low low low low low low low low low low
lower lower lower lower
lowest lowest lowest
high high high high high high high high
higher higher higher higher
highest highest highest
"""

def build_word_freq(text):
    words = re.findall(r'\w+', text.lower())
    word_freq = defaultdict(int)
    for word in words:
        word_freq[word] += 1
    return word_freq

def build_initial_vocab(word_freq):
    vocab = set()
    for word in word_freq.keys():
        for char in word:
            vocab.add(char)
    
    # Add common substrings
    subword_counts = defaultdict(int)
    for word, freq in word_freq.items():
        for i in range(len(word)):
            for j in range(i+2, min(i+5, len(word)+1)):
                subword = word[i:j]
                subword_counts[subword] += freq
    
    for subword, freq in subword_counts.items():
        if freq >= 3:
            vocab.add(subword)
    
    return list(vocab)

def get_token_freqs(vocab, word_freq):
    token_freqs = defaultdict(int)
    vocab_set = set(vocab)
    
    for word, freq in word_freq.items():
        # Greedy longest match
        tokens = []
        i = 0
        while i < len(word):
            found = False
            for j in range(len(word), i, -1):
                if word[i:j] in vocab_set:
                    tokens.append(word[i:j])
                    i = j
                    found = True
                    break
            if not found:
                tokens.append(word[i])
                i += 1
        
        for token in tokens:
            token_freqs[token] += freq
    
    return token_freqs

def compute_loss(word_freq, vocab):
    token_freqs = get_token_freqs(vocab, word_freq)
    total_freq = sum(token_freqs.values())
    
    loss = 0
    for word, freq in word_freq.items():
        tokens = []
        i = 0
        vocab_set = set(vocab)
        while i < len(word):
            found = False
            for j in range(len(word), i, -1):
                if word[i:j] in vocab_set:
                    tokens.append(word[i:j])
                    i = j
                    found = True
                    break
            if not found:
                tokens.append(word[i])
                i += 1
        
        word_loss = 0
        for token in tokens:
            prob = token_freqs.get(token, 1) / total_freq
            word_loss += -math.log(prob)
        loss += freq * word_loss
    
    return loss

def unigram_training(word_freq, initial_vocab, target_size):
    vocab = set(initial_vocab)
    chars = set()
    for word in word_freq.keys():
        for char in word:
            chars.add(char)
    
    print(f"Initial vocabulary: {len(vocab)} tokens")
    
    while len(vocab) > target_size:
        current_loss = compute_loss(word_freq, vocab)
        
        impact = {}
        for token in list(vocab):
            if token in chars:
                continue
            test_vocab = vocab - {token}
            test_loss = compute_loss(word_freq, test_vocab)
            impact[token] = test_loss - current_loss
        
        sorted_tokens = sorted(impact.items(), key=lambda x: x[1])
        num_to_remove = max(1, int(len(vocab) * 0.15))
        
        for token, _ in sorted_tokens[:num_to_remove]:
            vocab.remove(token)
        
        print(f"Vocab: {len(vocab)}, Loss: {current_loss:.2f}")
    
    return list(vocab)

def tokenize_word(word, vocab):
    tokens = []
    i = 0
    vocab_set = set(vocab)
    while i < len(word):
        found = False
        for j in range(len(word), i, -1):
            if word[i:j] in vocab_set:
                tokens.append(word[i:j])
                i = j
                found = True
                break
        if not found:
            tokens.append(word[i])
            i += 1
    return tokens

word_freq = build_word_freq(corpus)

print("CORPUS:")
for word, freq in sorted(word_freq.items(), key=lambda x: -x[1]):
    print(f"  {word}: {freq}")

initial_vocab = build_initial_vocab(word_freq)
print(f"\nInitial vocabulary: {len(initial_vocab)} tokens")

final_vocab = unigram_training(word_freq, initial_vocab, 20)

print(f"\nFINAL VOCABULARY ({len(final_vocab)} tokens):")
print(sorted(final_vocab))

print("\nTOKENIZATION RESULTS")
print("=" * 50)

test_words = ["low", "lower", "lowest", "high", "higher", "highest"]

for word in test_words:
    tokens = tokenize_word(word, final_vocab)
    print(f"{word:10} -> {' + '.join(tokens)}")

print("\nTOKEN FREQUENCIES")
print("=" * 50)
token_freqs = get_token_freqs(final_vocab, word_freq)
for token, freq in sorted(token_freqs.items(), key=lambda x: -x[1])[:10]:
    print(f"  {token}: {freq}")

CORPUS:
  low: 10
  high: 8
  lower: 4
  higher: 4
  lowest: 3
  highest: 3

Initial vocabulary: 39 tokens
Initial vocabulary: 39 tokens
Vocab: 34, Loss: 81.17
Vocab: 29, Loss: 59.95
Vocab: 25, Loss: 59.95
Vocab: 22, Loss: 59.95
Vocab: 19, Loss: 59.95

FINAL VOCABULARY (19 tokens):
['e', 'er', 'est', 'g', 'h', 'he', 'her', 'high', 'i', 'ig', 'l', 'low', 'o', 'r', 's', 'st', 't', 'w', 'wes']

TOKENIZATION RESULTS
low        -> low
lower      -> low + er
lowest     -> low + est
high       -> high
higher     -> high + er
highest    -> high + est

TOKEN FREQUENCIES
  low: 17
  high: 15
  er: 8
  est: 6


### SentencePiece Tokenisation  

In [45]:
!pip install sentencepiece

  Obtaining dependency information for sentencepiece from https://files.pythonhosted.org/packages/c2/83/4cfb393e287509fc2155480b9d184706ef8d9fa8cbf5505d02a5792bf220/sentencepiece-0.2.1-cp313-cp313-macosx_10_13_x86_64.whl.metadata
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 170.2 kB/s eta 0:00:0000:0100:01

[notice] A new release of pip is available: 23.2.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [49]:
import sentencepiece as spm
import os

# Define training options
options = dict(
    input="./files/luckin_coffee_transcript_cleaned.txt",             
    input_format="text",
    model_prefix="custom500",   
    model_type="unigram",       
    vocab_size=2268,  # The vocab size depends on the training data, in my case here i have 443 sentences and 30,575, it is the final size after EM pruning      
    normalization_rule_name="identity",
    remove_extra_whitespaces=False,
    input_sentence_size=200000000,
    max_sentence_length=4192,
    seed_sentencepiece_size=1000000,
    shuffle_input_sentence=True,
    character_coverage=0.99995,
    byte_fallback=True,
    split_digits=True,
    split_by_unicode_script=True,
    split_by_whitespace=True,
    split_by_number=True,
    max_sentencepiece_length=16,
    add_dummy_prefix=True,
    allow_whitespace_only_pieces=True,
    unk_id=0,
    bos_id=1,
    eos_id=2,
    pad_id=-1,
    num_threads=os.cpu_count(),
)

# Train the SentencePiece model
spm.SentencePieceTrainer.train(**options)

# Load and use the trained model
sp = spm.SentencePieceProcessor(model_file='custom500.model')

# Encode a sentence
encoded = sp.encode("In the first quarter, total net revenues grew 35% year-over-year")
print("Encoded:", encoded)

decoded = sp.decode(encoded)
print("Decoded:", decoded)

Encoded: [312, 270, 332, 305, 260, 267, 350, 331, 361, 753, 647, 259, 284, 288, 291, 304, 264, 338, 264, 328]
Decoded: In the first quarter, total net revenues grew 35% year-over-year


sentencepiece_trainer.cc(78) LOG(INFO) Starts training with : 
trainer_spec {
  input: ./files/luckin_coffee_transcript_cleaned.txt
  input_format: text
  model_prefix: custom500
  model_type: UNIGRAM
  vocab_size: 2268
  self_test_sample_size: 0
  character_coverage: 0.99995
  input_sentence_size: 200000000
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 8
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 1
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 1
  required_chars: 
  byte_fallback: 1
  vocabulary_output_piece_score: 1
  train_extremely_large_corpus: 0
  seed_sentencepieces_file: 
  hard_vocab_limit: 1
  use_all_vocab: 0
  unk_id: 0
  bos_id: 1
  eos_id: 2
  pad_id: -1
  unk_piece: <unk>
  bos_piece: <s>
  eos_piece: </s>
  pad_piece: <pad>
  unk_surface: